# FastWAM Official vs StarVLA Parity on Colab A100

This notebook is a thin orchestrator. Reusable parity logic lives in `examples/simBenchmarks/LIBERO/eval_files/fastwam_parity/`.

Runtime output directory: `/content/fastwam_parity`. Google Drive is not used.

In [ ]:
import os, subprocess, textwrap, json, pathlib
import torch

assert torch.cuda.is_available(), "A100 GPU runtime required"
props = torch.cuda.get_device_properties(0)
print({"gpu": props.name, "total_vram_gib": props.total_memory / 1024**3})
assert "A100" in props.name, f"Expected A100, got {props.name}"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["FLASH_ATTENTION_FORCE_DISABLE"] = "1"

In [ ]:
RUNTIME = pathlib.Path('/content/fastwam_parity')
RUNTIME.mkdir(parents=True, exist_ok=True)
%cd /content/fastwam_parity

STARVLA_REPO = 'https://github.com/okdmme/FastWAM_StarVLA_placement.git'
WORK_BRANCH = 'fastwam-colab-parity-repro-20260813'
START_REVISION = '374607be1e26d700f36b8a6f3f9fd30c018af79e'
OFFICIAL_REVISION = '45d8e1458921d83f8ad6cf9ce993d371208dabd0'

if not pathlib.Path('starVLA/.git').exists():
    subprocess.run(['git', 'clone', '--branch', WORK_BRANCH, STARVLA_REPO, 'starVLA'], check=True)
%cd /content/fastwam_parity/starVLA
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run(['git', 'merge-base', '--is-ancestor', START_REVISION, 'HEAD'], check=True)
print({'start_revision': START_REVISION, 'official_revision': OFFICIAL_REVISION})

In [ ]:
# Important package pins for reference-mode reproducibility.
!python -m pip install -q --upgrade pip
!python -m pip install -q \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  diffusers==0.34.0 transformers==4.51.3 accelerate==1.7.0 \
  safetensors==0.5.3 huggingface_hub==0.32.4 omegaconf==2.3.0 \
  pillow==11.2.1 numpy==1.26.4 sentencepiece==0.2.0 einops==0.8.1
!python -m pip install -q -e .
!python -m pip install -q -e /content/fastwam_parity/FastWAM_official || true

In [ ]:
# Download StarVLA-side diffusers encoders and released FastWAM checkpoint/stat files.
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='Wan-AI/Wan2.2-TI2V-5B-Diffusers',
    local_dir='playground/Pretrained_models/Wan-AI/Wan2.2-TI2V-5B-Diffusers',
    allow_patterns=['model_index.json','scheduler/**','tokenizer/**','text_encoder/**','vae/**'],
)
snapshot_download(
    repo_id='yuanty/fastwam',
    local_dir='checkpoints/fastwam_release',
    allow_patterns=['libero_uncond_2cam224.pt','libero_uncond_2cam224_dataset_stats.json'],
)
print('downloads ready')

In [ ]:
# Official FastWAM is cloned/pinned by the orchestrator and run sequentially before StarVLA.
!python -m examples.simBenchmarks.LIBERO.eval_files.fastwam_parity.colab_orchestrator \
  --runtime-dir /content/fastwam_parity \
  --starvla-dir /content/fastwam_parity/starVLA \
  --num-inference-steps 10 \
  --seed 7 \
  --dtype bfloat16

In [ ]:
import json, pathlib
comparison_path = pathlib.Path('/content/fastwam_parity/comparison.json')
comparison = json.loads(comparison_path.read_text())
print('num stages:', len(comparison))
for row in comparison[-5:]:
    print(row['stage'], row['torch_equal'], row['max_abs_error'], row['relative_l2_error'])
print('archive:', '/content/fastwam_parity/fastwam_parity_results.zip')